# Malfunction Classification: SL/SH Reporting-Dishonesty Types

**Notebook 09 of the spatiotemporal air quality anomaly detection pipeline**

---

This notebook classifies a station's deviation pattern by type rather than merely flagging
its presence: at every hourly slice, a station's reading is tested against a Tukey fence
built from its active neighbours, and the accumulated pattern of falling below or above that
fence — systematically, occasionally, or severely — determines whether a station is
classified as under-reporting, over-reporting, or malfunctioning outright, following the
framework of Chen et al. (2018).

| | |
|---|---|
| **Input** | `countries/anomaly_*.csv`, `config/params.yml` |
| **Output** | `station_suspicion_slsh.csv` |
| **Downstream** | the consensus notebook (`consensus.ipynb`) — cross-method consensus |


## Contents

0. **Method Context**
   - 0.1 Objective
   - 0.2 Approach and Principles
   - 0.3 Inputs and Outputs
1. **Environment and Data**
   - 1.1 Configuration
   - 1.2 Input Loading and Validation
2. **Hourly Slicing and the Value Matrix**
   - 2.1 Hourly Time Slices
   - 2.2 Station-by-Slice Value Matrix
3. **Spatial Neighbour Definition**
   - 3.1 Adjacency Matrix
4. **Per-Slice Tukey Fence Classification**
   - 4.1 The Tukey Fence
   - 4.2 Per-Station Detection Counts
5. **Cross-Slice Classification**
   - 5.1 Minimum Assessed Slices
   - 5.2 Systematic and Severe Deviation Ratios
6. **Suspicion Scoring and Flagging**
   - 6.1 Suspicion Score
   - 6.2 Flagging and Direction
7. **Results and Handoff**
   - 7.1 Flagged Stations
   - 7.2 Consensus-Ready Output
   - 7.3 Output Validation
8. **Findings and Limitations**

---

Terminology

- The **Tukey fence** is the pair of bounds, set at 1.5 interquartile ranges beyond the
  first and third quartiles, conventionally used to identify outliers in a reference
  distribution.
- **Φ_SL** and **Φ_SH** are the share of a station's assessed slices in which its reading
  fell below or above the Tukey fence built from its neighbours, following Chen et al.
  (2018).
- **Systematic** deviation is a Φ ratio exceeding one third of assessed slices;
  **severe malfunction** is a Φ ratio exceeding two thirds, both thresholds from Chen et al.
  (2018).
- **SL** (systematically low) and **SH** (systematically high) denote the direction of a
  station's classified deviation.


## 0. Method Context

### 0.1 Objective

Every method so far in this pipeline answers a version of the question "is this station
anomalous". None of them answers the question a CREA report ultimately needs to make
actionable: what *kind* of reporting dishonesty is suggested by the pattern. A station that
occasionally reads far below its neighbours calls for a different response than one that
occasionally reads far above them, and a station whose deviation is nearly constant calls for
a different response again from one whose deviation is intermittent.

Chen et al. (2018) provide a framework for exactly this distinction: a Tukey fence applied at
every time slice identifies whether a station's reading is unusually low or unusually high
relative to its neighbours at that moment, and the accumulated proportion of slices falling
on each side, benchmarked against literature-anchored thresholds, classifies the station's
overall pattern as systematic under-reporting, systematic over-reporting, severe malfunction,
or normal.

Two objectives follow:

1. **Classify the direction and severity of a station's deviation, not merely detect its
   presence.** <br>The classification itself — systematic under-report, systematic
   over-report, or severe malfunction — is the deliverable a policy-facing report can act on
   directly.</br>

2. **Apply literature-anchored thresholds rather than dataset-fitted ones.** <br>The
   boundaries between normal, systematic, and severe are taken from Chen et al. (2018) rather
   than derived from this dataset's own distribution, since the classification is meant to
   travel — comparable across datasets and future CREA analyses — rather than to be
   optimised for this run alone.</br>


### 0.2 Approach and Principles

Four decisions govern the implementation, several shared with Notebook 06 since both operate
on the same hourly-sliced spatial comparison.

1. **The same spatial and temporal infrastructure as Notebook 06.** <br>Hourly slicing,
   the adjacency structure, and the minimum-assessed-slices bar are identical in
   construction to Notebook 06, since both methods rest on the same underlying
   comparison — a station against its actively reporting neighbours at a given hour — before
   diverging in how that comparison is scored.</br>

2. **Tukey fences, not Z-scores.** <br>Where Notebook 06 standardises deviation with a
   modified Z-score, this notebook follows Chen et al. (2018) directly and uses the
   classical Tukey fence — quartiles and the interquartile range — since the classification
   thresholds this method applies (Sections 5 and 6) are defined by that paper specifically
   in terms of the Tukey construction, not a Z-score.</br>

3. **Classification thresholds are literature anchors, not derived values.** <br>The
   systematic (1/3) and severe (2/3) boundaries come directly from Chen et al. (2018) and are
   not tested for sensitivity against this dataset the way other thresholds in this pipeline
   are, since the value of this method rests on applying a known, external framework rather
   than calibrating a new one.</br>

4. **The minimum-assessed-slices bar is shared with Notebook 06.** <br>Both methods
   aggregate a proportion across hourly slices, and the sampling-stability reasoning behind
   how many slices a stable proportion requires is identical regardless of which proportion
   is being estimated. The value Notebook 06 derived is reused here rather than re-derived,
   consistent with how the spatial radius is shared rather than independently fitted by every
   spatial method in this pipeline.</br>


### 0.3 Inputs and Outputs

| Direction | Artifact | Contents |
|---|---|---|
| **In** | `countries/anomaly_*.csv` | Per-country analytical datasets (Notebook 02) |
| **In** | `config/params.yml` | Radius, minimum neighbours, shared minimum-slices bar |
| **Out** | `station_suspicion_slsh.csv` | Per-station Φ ratios, classification, suspicion score, flag, and direction |

The output schema matches the other detection notebooks — `location_id`, `country`,
`suspicion_score`, `flagged`, `direction` — with one addition specific to this method: a
`classification` column giving the Chen et al. (2018) category directly.

This notebook reads no output from any other detection notebook, including Notebook 06: the
two share a spatial and temporal construction by design, but each computes it independently
from the raw observational record and reaches its own verdict by a distinct scoring rule.


## 1. Environment and Data

### 1.1 Configuration

The spatial parameters — radius and minimum neighbour count — are the same values Notebook
03 and Notebook 06 use. The minimum-assessed-slices bar is the value Notebook 06 derived by
sensitivity analysis, reused here under the shared-parameter reasoning set out in Section
0.2. The Tukey and classification constants are literature anchors, declared as named
constants with their citation rather than drawn from configuration, since they are not
subject to dataset-specific derivation.


In [1]:
import glob
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_style import table_style

# ── Configuration ────────────────────────────────────────────────────
CONFIG_PATH = Path("../config/params.yml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

REQUIRED_SECTIONS = ["meta", "spatial", "tsad", "paths", "colors_map"]
_missing = [s for s in REQUIRED_SECTIONS if s not in CONFIG]
if _missing:
    raise KeyError(f"Missing configuration section(s): {_missing}")

PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
FIGURE_DIR    = PROCESSED_DIR.parent.parent / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_COLOURS = CONFIG["colors_map"]
COUNTRY_ORDER   = ["China", "Germany", "India", "USA"]
RANDOM_SEED     = CONFIG["meta"]["random_seed"]

MIN_NEIGHBORS = CONFIG["spatial"]["min_neighbors"]
DEFAULT_RADIUS = CONFIG["spatial"]["radius_km"]["default"]
MIN_SLICES = CONFIG["tsad"].get("tsad_min_slices", 100)

def radius_for(country: str) -> float:
    r = CONFIG["spatial"]["radius_km"].get(country)
    return r if r is not None else DEFAULT_RADIUS

# Literature-anchored constants (Chen et al., 2018) — not derived from this dataset
TUKEY_R          = 1.5    # Tukey fence multiplier, Eq. 1-2
PHI_SYSTEMATIC   = 1 / 3  # systematic-deviation boundary, Eq. 5-6
PHI_SEVERE       = 2 / 3  # severe-malfunction boundary, Eq. 5-6

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)

parameters = pd.DataFrame([
    ("Minimum neighbours", MIN_NEIGHBORS, "Shared with Notebooks 03, 08 (NB02 §7.3)"),
    ("Minimum assessed slices", MIN_SLICES, "Shared with Notebook 06 (sensitivity-derived)"),
    ("Tukey fence multiplier", TUKEY_R, "Chen et al. (2018), Eq. 1-2"),
    ("Systematic boundary (Φ)", round(PHI_SYSTEMATIC, 3), "Chen et al. (2018), Eq. 5-6"),
    ("Severe boundary (Φ)", round(PHI_SEVERE, 3), "Chen et al. (2018), Eq. 5-6"),
], columns=["Parameter", "Value", "Source"]).set_index("Parameter")

display(table_style(parameters))


,Parameter,Value,Source
0,Minimum neighbours,3.000000,"Shared with Notebooks 03, 08 (NB02 §7.3)"
1,Minimum assessed slices,100.000000,Shared with Notebook 06 (sensitivity-derived)
2,Tukey fence multiplier,1.500000,"Chen et al. (2018), Eq. 1-2"
3,Systematic boundary (Φ),0.333000,"Chen et al. (2018), Eq. 5-6"
4,Severe boundary (Φ),0.667000,"Chen et al. (2018), Eq. 5-6"


<u>Interpretation</u>

The spatial and temporal-sampling parameters are reused from where they were already
validated; the classification thresholds are declared as literature constants rather than
configuration values, since — unlike a completeness bar or a contamination rate — they are
not properties of this dataset to be derived, but a fixed external framework this notebook
applies to it.


### 1.2 Input Loading and Validation

The per-country files written by Notebook 02 are the sole input, floored to the calendar
hour on load exactly as in Notebook 06.


In [2]:
country_files = sorted(glob.glob(str(PROCESSED_DIR / "countries" / "anomaly_*.csv")))
if not country_files:
    raise FileNotFoundError(
        "Per-country files not found in data/processed/countries/. "
        "Run 02_exploratory_data_analysis.ipynb (Section 8) to completion first."
    )

meas = pd.concat([pd.read_csv(f) for f in country_files], ignore_index=True)
meas["date"] = pd.to_datetime(meas["date"], utc=True, errors="coerce")

REQUIRED_COLUMNS = {"location_id", "country", "value", "date", "latitude", "longitude"}
_absent = REQUIRED_COLUMNS - set(meas.columns)
if _absent:
    raise KeyError(f"Fields required for SL/SH classification are absent: {_absent}")

meas["hour_slice"] = meas["date"].dt.floor("h")
meas = meas.dropna(subset=["hour_slice", "value"])

loaded = pd.DataFrame([
    ("Observations",  f"{len(meas):,}"),
    ("Stations",      f"{meas['location_id'].nunique():,}"),
    ("Hourly slices", f"{meas['hour_slice'].nunique():,}"),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(loaded))


/var/folders/p4/dvtmgspd2kj00jkgk5zfym840000gn/T/ipykernel_32201/2359998775.py:8: DtypeWarning: Columns (0: review_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  meas = pd.concat([pd.read_csv(f) for f in country_files], ignore_index=True)


,Property,Value
0,Observations,"40,505,358"
1,Stations,"3,150"
2,Hourly slices,"26,280"


<u>Interpretation</u>

The eligible per-country record loads with the same hourly slicing Notebook 06 applies,
built independently from the raw record rather than imported from it.


## 2. Hourly Slicing and the Value Matrix

The dense station-by-slice matrix this notebook operates on is constructed identically to
Notebook 06's, since both methods need the same representation of who reported what at
every hour.


### 2.1 Hourly Time Slices

The slice index is the sorted set of distinct hours present in the eligible record.


In [3]:
stations = (meas.groupby("location_id")
            .agg(country=("country", "first"),
                 latitude=("latitude", "first"),
                 longitude=("longitude", "first"))
            .reset_index())
N = len(stations)

hour_index = np.sort(meas["hour_slice"].unique())
T = len(hour_index)

display(table_style(pd.DataFrame([
    ("Stations", f"{N:,}"),
    ("Hourly slices", f"{T:,}"),
], columns=["Property", "Value"]).set_index("Property")))


,Property,Value
0,Stations,"3,150"
1,Hourly slices,"26,280"


<u>Interpretation</u>

The matrix dimensions match those in Notebook 06, as expected — both notebooks slice the
same eligible record the same way.


### 2.2 Station-by-Slice Value Matrix

Observations are pivoted into a dense matrix, one row per hourly slice and one column per
station, with a missing cell wherever a station did not report at that hour.


In [4]:
pivot = (meas.pivot_table(index="hour_slice", columns="location_id",
                          values="value", aggfunc="mean")
          .reindex(index=hour_index, columns=stations["location_id"]))
V = pivot.values

display(table_style(pd.DataFrame([
    ("Matrix shape", f"{V.shape[0]:,} × {V.shape[1]:,}"),
    ("Filled cells", f"{np.isfinite(V).sum():,} ({np.isfinite(V).mean()*100:.1f}%)"),
], columns=["Property", "Value"]).set_index("Property")))


,Property,Value
0,Matrix shape,"26,280 × 3,150"
1,Filled cells,"40,505,358 (48.9%)"


<u>Interpretation</u>

The value matrix is the shared representation both this notebook and Notebook 06 build from
the same raw record; from here the two methods diverge in how they score each station's
readings against its neighbours.


## 3. Spatial Neighbour Definition

The adjacency structure is identical in construction to Notebook 03 and Notebook 06: same
country, within radius, excluding a station from its own neighbour set. It is rebuilt here
independently rather than imported, consistent with the pipeline's method-independence
principle.


### 3.1 Adjacency Matrix



In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres between coordinate arrays."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


lat, lon, ctry = stations["latitude"].values, stations["longitude"].values, stations["country"].values
adjacency = np.zeros((N, N), dtype=bool)

for i in range(N):
    d = haversine_km(lat[i], lon[i], lat, lon)
    same_country = ctry == ctry[i]
    adjacency[i] = (d <= radius_for(ctry[i])) & (d > 0) & same_country

display(table_style(pd.DataFrame([
    ("Adjacency matrix shape", f"{N} × {N}"),
    ("Stations with ≥ minimum neighbours", f"{(adjacency.sum(axis=1) >= MIN_NEIGHBORS).sum()} / {N}"),
], columns=["Property", "Value"]).set_index("Property")))


,Property,Value
0,Adjacency matrix shape,3150 × 3150
1,Stations with ≥ minimum neighbours,2754 / 3150


<u>Interpretation</u>

The adjacency structure reproduces the same neighbour relationships used throughout this
pipeline's spatial methods, providing the reference set the Tukey fence in Section 4 is built
from at every slice.


## 4. Per-Slice Tukey Fence Classification

At every hourly slice, a station's reading is tested against a Tukey fence constructed from
its actively reporting neighbours at that same hour — the classical outlier-detection
construction Chen et al. (2018) build their framework on.


### 4.1 The Tukey Fence

For a station with enough active neighbours at a slice, the first and third quartiles of
those neighbours' readings define the interquartile range, and the fence extends 1.5 times
that range beyond each quartile. A reading below the lower fence is a candidate low
deviation for that slice; above the upper fence, a candidate high deviation. The neighbour
set automatically excludes the station itself, since it is not a member of its own
neighbourhood — the same leave-one-out principle applied throughout this pipeline.


In [6]:
def per_station_tukey_flags(own_col: np.ndarray, neighbour_cols: np.ndarray):
    """Leave-one-out Tukey fence test of one station against its neighbours, per slice.

    Returns three boolean/count arrays of shape (T,): below the lower fence, above
    the upper fence, and whether the slice had enough active neighbours to test at all.
    """
    active_count = np.isfinite(neighbour_cols).sum(axis=1)
    q1 = np.nanpercentile(neighbour_cols, 25, axis=1)
    q3 = np.nanpercentile(neighbour_cols, 75, axis=1)
    iqr = q3 - q1
    lower_fence = q1 - TUKEY_R * iqr
    upper_fence = q3 + TUKEY_R * iqr

    valid = np.isfinite(own_col) & (active_count >= MIN_NEIGHBORS)
    below = valid & (own_col < lower_fence)
    above = valid & (own_col > upper_fence)
    return below, above, valid


sl_count = np.zeros(N, dtype=int)
sh_count = np.zeros(N, dtype=int)
n_assessed = np.zeros(N, dtype=int)

for i in range(N):
    neighbour_idx = np.where(adjacency[i])[0]
    if len(neighbour_idx) < MIN_NEIGHBORS:
        continue
    below, above, valid = per_station_tukey_flags(V[:, i], V[:, neighbour_idx])
    sl_count[i] = below.sum()
    sh_count[i] = above.sum()
    n_assessed[i] = valid.sum()

display(table_style(pd.DataFrame([
    ("Stations with any assessed slice", f"{(n_assessed > 0).sum():,} / {N:,}"),
    ("Total low-fence detections (candidate SL)", f"{int(sl_count.sum()):,}"),
    ("Total high-fence detections (candidate SH)", f"{int(sh_count.sum()):,}"),
], columns=["Property", "Value"]).set_index("Property")))


/Users/endarlani/anaconda3/envs/crea/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


,Property,Value
0,Stations with any assessed slice,"2,754 / 3,150"
1,Total low-fence detections (candidate SL),"1,529,322"
2,Total high-fence detections (candidate SH),"2,569,083"


<u>Interpretation</u>

Every slice contributes an independent Tukey test wherever a station and enough of its
neighbours are simultaneously active — the same sparsity consideration as Notebook 06
applies here. The raw detection counts are not yet the classification; Section 5 converts
them into the Φ ratios Chen et al. (2018) classify stations by.


### 4.2 Per-Station Detection Counts

Before the ratios are computed, the raw counts are examined per country, since a network
with systematically wider or narrower neighbour dispersion will produce systematically
different raw detection volumes even before any station-level classification.


In [7]:
detection_summary = pd.DataFrame({
    "location_id": stations["location_id"].values,
    "country": stations["country"].values,
    "n_slices_assessed": n_assessed,
    "sl_count": sl_count,
    "sh_count": sh_count,
})

display(table_style(detection_summary.groupby("country")
                    [["n_slices_assessed", "sl_count", "sh_count"]].sum()))


,country,n_slices_assessed,sl_count,sh_count
0,China,12034683,689161,942967
1,Germany,5618372,269493,438227
2,India,6175595,183197,457584
3,USA,10277575,387471,730305


<u>Interpretation</u>

The per-country totals give a first read on which networks carry more raw fence detections
in aggregate, but this is not yet informative about any individual station: a large total can
come from many stations deviating a little or a few stations deviating persistently, and
only the per-station ratios in Section 5 distinguish the two.


## 5. Cross-Slice Classification

The raw per-slice detection counts are converted into station-level ratios, and those ratios
are classified against the Chen et al. (2018) boundaries.


### 5.1 Minimum Assessed Slices

As in Notebook 06, a ratio computed from few assessed slices is unstable and is withheld
rather than reported. The bar is the same value Notebook 06 derived, reused under the
shared-parameter reasoning in Section 0.2.


In [8]:
detection_summary["assessed"] = detection_summary["n_slices_assessed"] >= MIN_SLICES

display(table_style(pd.DataFrame([
    ("Minimum assessed slices", MIN_SLICES),
    ("Stations assessed", f"{int(detection_summary['assessed'].sum()):,} / {N:,}"),
], columns=["Property", "Value"]).set_index("Property")))


,Property,Value
0,Minimum assessed slices,100
1,Stations assessed,"2,754 / 3,150"


<u>Interpretation</u>

Reusing Notebook 06's derived bar rather than re-deriving it keeps the two methods'
eligibility criteria identical, so a station judged assessed by one is judged assessed by
the other on the same basis.


### 5.2 Systematic and Severe Deviation Ratios

Φ_SL and Φ_SH are the share of a station's assessed slices in which it fell below or above
the Tukey fence respectively. Classification follows Chen et al. (2018) directly: a ratio
beyond the severe boundary is severe malfunction regardless of direction; beyond the
systematic boundary but not the severe one is systematic under- or over-reporting; below the
systematic boundary is normal.


In [9]:
with np.errstate(invalid="ignore", divide="ignore"):
    phi_sl = np.where(n_assessed > 0, sl_count / n_assessed, np.nan)
    phi_sh = np.where(n_assessed > 0, sh_count / n_assessed, np.nan)

def classify(sl: float, sh: float) -> str:
    if np.isnan(sl):
        return "unassessed"
    if sl > PHI_SEVERE or sh > PHI_SEVERE:
        return "severe_malfunction"
    if sl > PHI_SYSTEMATIC:
        return "systematic_under_report"
    if sh > PHI_SYSTEMATIC:
        return "systematic_over_report"
    return "normal"

slsh = detection_summary.copy()
slsh["phi_sl"] = phi_sl
slsh["phi_sh"] = phi_sh
slsh["classification"] = [classify(sl, sh) for sl, sh in zip(phi_sl, phi_sh)]

display(table_style(slsh.loc[slsh["assessed"], "classification"]
                    .value_counts().rename("stations").to_frame()))
display(table_style(slsh[slsh["assessed"] & ~slsh["classification"].isin(["normal"])]
                    .groupby(["country", "classification"]).size()
                    .rename("stations").to_frame()))


,classification,stations
0,normal,2703
1,systematic_under_report,26
2,systematic_over_report,25


,country,classification,stations
0,China,systematic_over_report,9
1,China,systematic_under_report,19
2,Germany,systematic_over_report,1
3,Germany,systematic_under_report,2
4,India,systematic_over_report,12
5,India,systematic_under_report,4
6,USA,systematic_over_report,3
7,USA,systematic_under_report,1


<u>Interpretation</u>

The classification counts give the first substantive answer this method is designed to
provide: not how many stations are anomalous, but how many of each specific type. The
per-country breakdown shows whether a particular failure mode concentrates in one network,
which is directly relevant to how a CREA report would frame a finding — a network dominated
by systematic under-reporting suggests a different investigation than one showing scattered
severe malfunctions.


## 6. Suspicion Scoring and Flagging

The classification itself is the primary output, but a continuous suspicion score is still
required for the consensus in the consensus notebook (`consensus.ipynb`) to combine this method with the others.


### 6.1 Suspicion Score

The suspicion score is the larger of a station's two Φ ratios — however dominant its
deviation is, in whichever direction it runs — giving a single continuous quantity on the
same higher-means-more-suspicious scale as every other method in this pipeline.


In [10]:
slsh["suspicion_score"] = np.nanmax(slsh[["phi_sl", "phi_sh"]].values, axis=1)

display(table_style(slsh[slsh["assessed"]].nlargest(10, "suspicion_score")
                    [["location_id", "country", "classification", "phi_sl", "phi_sh"]]
                    .reset_index(drop=True).round(3)))


/var/folders/p4/dvtmgspd2kj00jkgk5zfym840000gn/T/ipykernel_32201/2703094554.py:1: RuntimeWarning: All-NaN slice encountered
  slsh["suspicion_score"] = np.nanmax(slsh[["phi_sl", "phi_sh"]].values, axis=1)


,location_id,country,classification,phi_sl,phi_sh
0,site_136,India,systematic_over_report,0.014000,0.612000
1,site_5669,India,systematic_over_report,0.018000,0.559000
2,3058a,China,systematic_over_report,0.011000,0.536000
3,site_1395,India,systematic_over_report,0.019000,0.517000
4,1564a,China,systematic_under_report,0.501000,0.040000
5,1857a,China,systematic_under_report,0.498000,0.049000
6,sta.de_dehe051,Germany,systematic_under_report,0.495000,0.012000
7,site_5855,India,systematic_under_report,0.471000,0.004000
8,2467a,China,systematic_under_report,0.463000,0.026000
9,airnow_060631010,USA,systematic_over_report,0.001000,0.445000


<u>Interpretation</u>

The highest-scoring stations are those whose classification rests on the largest Φ ratio,
which — combined with the `classification` column carried in the output — lets a consumer of
this file rank stations by severity while still reading off the specific type of deviation
each one represents.


### 6.2 Flagging and Direction

A station is flagged if it is assessed and classified as anything other than normal.
Direction follows which ratio dominates, using the same SL/SH vocabulary as Notebook 03 and
Notebook 06.


In [11]:
slsh["flagged"] = slsh["assessed"] & slsh["classification"].isin(
    ["systematic_under_report", "systematic_over_report", "severe_malfunction"])

def classify_direction(row):
    if not row["flagged"]:
        return "normal"
    return "SL_under_report" if row["phi_sl"] >= row["phi_sh"] else "SH_over_report"

slsh["direction"] = slsh.apply(classify_direction, axis=1)

flag_summary = pd.DataFrame([
    ("Stations assessed", int(slsh["assessed"].sum())),
    ("Flagged as suspicious", int(slsh["flagged"].sum())),
], columns=["Category", "Stations"]).set_index("Category")

display(table_style(flag_summary))


,Category,Stations
0,Stations assessed,2754
1,Flagged as suspicious,51


<u>Interpretation</u>

Flagging here is a direct consequence of the Chen et al. (2018) classification rather than a
separately tuned cutoff, since the systematic and severe boundaries already determine which
stations warrant attention. This is a structural difference from every other threshold in
this pipeline, where a cutoff is derived or anchored and then examined for sensitivity: here
the literature framework supplies both the classification and the flag in one step.


## 7. Results and Handoff

### 7.1 Flagged Stations

Flagged stations are shown with their full classification and both Φ ratios, so the type and
strength of each deviation is visible directly rather than collapsed into a single score.


In [12]:
ranked = (slsh[slsh["flagged"]]
          .sort_values("suspicion_score", ascending=False)
          [["location_id", "country", "classification", "phi_sl", "phi_sh", "n_slices_assessed"]]
          .reset_index(drop=True)
          .round(3))

display(table_style(ranked.head(15)))


,location_id,country,classification,phi_sl,phi_sh,n_slices_assessed
0,site_136,India,systematic_over_report,0.014000,0.612000,5449
1,site_5669,India,systematic_over_report,0.018000,0.559000,7653
2,3058a,China,systematic_over_report,0.011000,0.536000,8567
3,site_1395,India,systematic_over_report,0.019000,0.517000,6344
4,1564a,China,systematic_under_report,0.501000,0.040000,8591
5,1857a,China,systematic_under_report,0.498000,0.049000,8384
6,sta.de_dehe051,Germany,systematic_under_report,0.495000,0.012000,7189
7,site_5855,India,systematic_under_report,0.471000,0.004000,244
8,2467a,China,systematic_under_report,0.463000,0.026000,8352
9,airnow_060631010,USA,systematic_over_report,0.001000,0.445000,15316


<u>Interpretation</u>

These stations carry a specific, literature-grounded classification rather than a generic
anomaly label — the distinction this method is built to provide. As with every method in this
pipeline, the classification is a candidate finding rather than a conclusion; its standing is
settled in the consensus notebook (`consensus.ipynb`), where agreement with methods reasoning from independent evidence
separates robust signals from method-specific artefacts.


### 7.2 Consensus-Ready Output

Every station is written, including those found normal or unassessed, so the consensus can
distinguish a station this method assessed and cleared from one it could not assess at all.


In [13]:
output = slsh[["location_id", "country", "n_slices_assessed", "phi_sl", "phi_sh",
               "classification", "suspicion_score", "assessed", "flagged", "direction"]].copy()

output["method"] = "slsh"
output = output.sort_values("suspicion_score", ascending=False, na_position="last")

OUTPUT_PATH = PROCESSED_DIR / "station_suspicion_slsh.csv"
output.to_csv(OUTPUT_PATH, index=False)

manifest = pd.DataFrame([
    ("Output file", OUTPUT_PATH.name),
    ("Stations written", f"{len(output):,}"),
    ("Flagged", f"{int(output['flagged'].sum()):,}"),
    ("Assessed", f"{int(output['assessed'].sum()):,}"),
    ("Consensus schema", "location_id, country, suspicion_score, flagged, direction (+ classification)"),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(manifest))


,Property,Value
0,Output file,station_suspicion_slsh.csv
1,Stations written,"3,150"
2,Flagged,51
3,Assessed,"2,754"
4,Consensus schema,"location_id, country, suspicion_score, flagged, direction (+ classification)"


<u>Interpretation</u>

The result carries both the shared consensus schema and the method-specific classification
label, so a consumer that only needs a flag can use one directly, while a consumer preparing
a report can read off the specific dishonesty type Chen et al.'s framework assigns.


### 7.3 Output Validation



In [14]:
reloaded = pd.read_csv(OUTPUT_PATH)

checks = pd.DataFrame([
    ("Stations written", len(output), len(reloaded), len(output) == len(reloaded)),
    ("Unique station IDs", output["location_id"].nunique(), reloaded["location_id"].nunique(),
     output["location_id"].nunique() == reloaded["location_id"].nunique()),
    ("Flagged count", int(output["flagged"].sum()), int(reloaded["flagged"].sum()),
     int(output["flagged"].sum()) == int(reloaded["flagged"].sum())),
    ("Consensus columns present", "yes",
     "yes" if {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns) else "no",
     {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns)),
], columns=["Check", "Computed", "Reloaded", "Pass"]).set_index("Check")

display(table_style(checks))

assert len(output) == len(reloaded), "Row count changed on write."
assert output["location_id"].duplicated().sum() == 0, "Duplicate station in output."
assert {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns), \
    "Consensus schema incomplete; the consensus notebook (`consensus.ipynb`) would fail on this file."


,Check,Computed,Reloaded,Pass
0,Stations written,3150,3150,True
1,Unique station IDs,3150,3150,True
2,Flagged count,51,51,True
3,Consensus columns present,yes,yes,True


<u>Interpretation</u>

The written file reconciles with the computed result and carries the consensus schema
intact, alongside the classification detail specific to this method.


## 8. Findings and Limitations

### Findings

1. **Classification, not merely detection, is this method's distinct contribution.** <br>Of
every method in this pipeline, only this one assigns a named dishonesty type to a flagged
station — systematic under-report, systematic over-report, or severe malfunction — directly
answering the question a CREA report needs to act on.</br>

2. **Thresholds are applied from the literature, not fitted to this dataset.** <br>The
Tukey multiplier and the systematic/severe boundaries are Chen et al. (2018)'s own values,
unmodified. This is a deliberate methodological choice: the classification is meant to be
comparable across datasets and future analyses, not optimised for this run.</br>

3. **Shared infrastructure with Notebook 06 does not compromise independence.** <br>The two
methods build identical hourly slicing and adjacency structures from the raw record, each
independently, and diverge entirely in how they score a station's deviation — Tukey fences
and literature classification here, modified Z-scores and a derived flagging rate there.
Agreement between them is still evidence from two distinct scoring rules, not a restatement
of one by the other.</br>

### Limitations

1. **The Tukey fence, like the modified Z-score, depends on active-neighbour count at each
slice.** <br>The same sparsity that limits Notebook 06 limits this method identically: most
hourly slices lack enough simultaneously active neighbours to support the comparison, which
is why the minimum-assessed-slices bar exists.</br>

2. **Chen et al.'s thresholds were developed for sensor malfunction, not necessarily
deliberate manipulation.** <br>The 1/3 and 2/3 boundaries were validated in a fault-detection
context; applying them to suspected intentional misreporting is an adaptation of the
framework's intended scope, not a re-derivation for this purpose, and is stated as such
rather than presented as a perfect match.</br>

3. **The neighbourhood is assumed correct at every slice.** <br>As in Notebook 06, a
regional bias affecting an entire local area at a given hour would not be distinguishable
from every affected station behaving normally relative to a displaced neighbourhood.</br>

4. **Unassessed and non-systematic deviation are not the same thing.** <br>A station
classified `unassessed` lacked enough assessed slices to classify at all; a station
classified `normal` was classified and found below the systematic boundary. The two must not
be conflated when reading the output.</br>

---

### Output

| File | Contents |
|---|---|
| `station_suspicion_slsh.csv` | Per-station Φ ratios, classification, suspicion score, flag, and direction |

**Next:** `10_dbscan.ipynb` — a local density-based detector operating on the shared station
feature profiles.

---

### References

Chen, X., et al. (2018). Detecting anomalous data in IoT-based air quality monitoring.
*IEEE Internet of Things Journal*, 5(2).
